# 三个可以立即检查的机制

标准库、CPU，无网络与模型密钥。约 15 分钟。小数据用于验证筛选、loss mask 的位置、版本检查，不代表真实记忆检索、RL 训练或数据库并发成绩。

## 1. 最相似的记忆为什么可能答错？
先猜 u1 当前与历史查询的结果。

In [ ]:
records = [
    {'user': 'u1', 'value': 'Python 3.9', 'from': '2025-01-01', 'until': '2026-01-01', 'similarity': .98},
    {'user': 'u1', 'value': 'Python 3.12', 'from': '2026-01-01', 'until': None, 'similarity': .86},
    {'user': 'u2', 'value': 'Python 3.13', 'from': '2026-01-01', 'until': None, 'similarity': .99},
]
def retrieve(user, date):
    candidates = [r for r in records if r['user'] == user and r['from'] <= date
                  and (r['until'] is None or date < r['until'])]
    return max(candidates, key=lambda r: r['similarity'])['value']

naive = max(records, key=lambda r: r['similarity'])['value']
current = retrieve('u1', '2026-10-11')
history = retrieve('u1', '2025-06-01')
assert (naive, current, history) == ('Python 3.13', 'Python 3.12', 'Python 3.9')
print('只按相似度：', naive, '当前 u1：', current, '历史 u1：', history)


## 2. 工具观察是否应计入策略生成损失？
这里只演示 masked negative log likelihood。完整 RL 还需要 advantage、rollout logprob、KL 与训练器；观察虽然被排除直接生成损失，仍可作为后续动作的条件上下文。

In [ ]:
import math
# 教学用位置级概率；真实训练要检查 shift 后的 labels、mask 与 logprob 对齐。
roles = ['action', 'action', 'observation', 'observation', 'answer']
probability = [.8, .6, .1, .2, .9]
mask = [1, 1, 0, 0, 1]
loss = sum(-math.log(p) * m for p, m in zip(probability, mask)) / sum(mask)
wrong_loss = sum(-math.log(p) for p in probability) / len(probability)
assert mask[2:4] == [0, 0]
assert abs(loss - .2797765635793423) < 1e-12
assert wrong_loss > loss
print('动作位置：', [i for i, m in enumerate(mask) if m])
print('正确 mask 的示意损失：', round(loss, 4), '混入观察：', round(wrong_loss, 4))


## 3. 旧 worker 还能提交状态吗？
这是顺序执行的版本检查示意。真实数据库要把检查与更新合成原子 CAS；外部工具写入还要独立处理幂等、查询与对账。

In [ ]:
state = {'version': 7, 'status': 'running'}
old_worker = state['version']
# 租约过期，worker B 取得新版本。
state['version'] = 8
new_worker = state['version']
def commit(worker_version, status):
    if worker_version != state['version']:
        return False
    state['status'] = status
    return True

assert commit(old_worker, 'succeeded') is False
assert state['status'] == 'running'
assert commit(new_worker, 'succeeded') is True
print('旧 worker 提交被拒绝，新 worker 提交成功：', state)


## 自己改一处

记忆：去掉 user 条件或时间条件，检查是哪条记录污染答案。Mask：错移一位并列出实际参与损失的位置。版本：去掉版本检查，观察旧 worker 覆盖状态。将原因写回网站练习。

这三个例子的全部代码单元已在本机执行并验证断言。